In [0]:
import sys
import delta
from pyspark.sql import functions as F

sys.path.insert(0, "../lib/")

import db

table = 'roles'
path = f"/Volumes/raw/tcc-climate-data/full_load/"

In [0]:
class IngestaoFullBronze:
    def __init__(self, table, idField):
        self.table = table
        self.tableName = f'bronze.climapetropolis.{table}'
        self.idField = idField
        self.query = db.import_query(f"{table}.sql")
        self.query = self.query.format(table=table)

    def read(self):
        df = spark.read.json(path)
        return df
    
    def transform(self, df):
        df.createOrReplaceTempView(self.table)
        df_transform = spark.sql(self.query)
        return df_transform
    
    def load(self, df):
        (df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(self.tableName))
        
    def auto(self):
        df = self.read()
        df_transform = self.transform(df)
        self.load(df_transform)

In [0]:
ingestao = IngestaoFullBronze(table = 'roles', idField = 'id')
ingestao.auto()

In [0]:
class IngestaoFullBronzeStreaming(IngestaoFullBronze):
       def __init__(self, table, idField, spark):
              super().__init__(table, idField)
              self.checkpoint_path = "/Volumes/raw/tcc-climate-data/full_load/checkpoints/" 
              self.spark = spark
       
       def read(self):
             return (self.spark.readStream
                               .format("cloudFiles")
                               .option("cloudFiles.format", "json")
                               .option("cloudFiles.schemaLocation", f"{self.checkpoint_path}/schema")
                               .option("cloudFiles.inferColumnTypes", "false")
                               .load(path))     

       def transform(self, df, batch_id):

            df.createOrReplaceTempView(self.table)

            df = self.spark.sql(self.query.format(table = self.table))      
            df = df.dropDuplicates([self.idField])
            
            delta_table = delta.DeltaTable.forName(self.spark, self.tableName)
            join_on = f"d.{self.idField} = t.{self.idField}"

            (delta_table.alias("d")
                        .merge(df.alias("t"), join_on)
                        .whenMatchedUpdateAll()
                        .whenNotMatchedInsertAll()
                        .execute())

       def save(self, df):
              (df.writeStream
                        .foreachBatch(self.transform)
                        .option("checkpointLocation", f"{self.checkpoint_path}/bronze_{self.table}")
                        .trigger(availableNow=True)
                        .start())

       def auto(self):

              df = self.read()
              self.save(df)

In [0]:
ingestao = IngestaoFullBronzeStreaming(table, 'id', spark)
ingestao.auto()

In [0]:
%sql
SELECT * FROM roles